#### Load Data

In [102]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from itertools import product
from openpyxl import Workbook

# load dataset
df = pd.read_csv("../data/coffee_shop_sales.csv")

# display information
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Shape: (149116, 11)

Columns:
['Transaction ID', 'Transaction Date', 'Transaction Time', 'Transaction Quantity', 'Store ID', 'Store Location', 'Product ID', 'Unit Price', 'Product Category', 'Product Type', 'Product Detail']


#### Add Quarter Columns

In [103]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])

/var/folders/53/vr8gs__x60sg207tybh25bv80000gn/T/ipykernel_81889/3594840833.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])


In [104]:
df["Quarter"] = df["Transaction Date"].dt.to_period("Q")

print(df["Quarter"].value_counts().sort_index())

Quarter
2023Q1    54902
2023Q2    94214
Freq: Q-DEC, Name: count, dtype: int64


In [105]:
for quarter in df["Quarter"].sort_values().unique():
    quarter_df = df[df["Quarter"] == quarter]

    print(
        quarter,
        quarter_df["Transaction Date"].min(),
        quarter_df["Transaction Date"].max(),
        len(quarter_df)
    )

2023Q1 2023-01-01 00:00:00 2023-03-31 00:00:00 54902
2023Q2 2023-04-01 00:00:00 2023-06-30 00:00:00 94214


#### Calculate Business Metrics

In [110]:
quarterly_results = {}

quarters = df["Quarter"].sort_values().unique()

for quarter in quarters:

    # filter data per quarter
    quarter_df = df[df["Quarter"] == quarter].copy()

    # metrics
    quarter_df["Revenue"] = (
    quarter_df["Transaction Quantity"] *
    quarter_df["Unit Price"]
    )
    #------------------------------------------
    total_revenue = quarter_df["Revenue"].sum()

    total_transactions = quarter_df["Transaction ID"].nunique()

    total_units = quarter_df["Transaction Quantity"].sum()

    average_transaction_value = (
        total_revenue / total_transactions
    )

    average_units_per_transaction = (
        total_units / total_transactions
    )
    # print(f"\nMetrics for {quarter}:")
    # print("Revenue:", total_revenue)
    # print("Transactions:", total_transactions)
    # print("Units:", total_units)
    # print("Average Transaction Value:", average_transaction_value)
    # print("Average Units per Transaction:", average_units_per_transaction)


    # product related metrics

    product_sales = (
        quarter_df.groupby("Product Type")["Revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    category_sales = (
        quarter_df.groupby("Product Category")["Revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    quarter_df["Revenue %"] = (
        quarter_df["Revenue"] /
        quarter_df["Revenue"].sum()
    )


    # metrics by product

    product_sales_metrics = (
        quarter_df.groupby("Product Detail")
        .agg(
            revenue=("Revenue", "sum"),
            units_sold=("Transaction Quantity", "sum"),
            transactions=("Transaction ID", "nunique")
        )
        .sort_values("revenue", ascending=False)
    )


    # revenue divided into various time periods

    quarter_df["Transaction Date"] = pd.to_datetime(
        quarter_df["Transaction Date"],
        errors="coerce"
    )

    sales_by_day = (
        quarter_df.groupby("Transaction Date")["Revenue"]
        .sum()
    )

    quarter_df["Transaction Time"] = pd.to_datetime(
        quarter_df["Transaction Time"],
        format="%H:%M:%S",
        errors="coerce"
    )


    # extract hour

    quarter_df["Transaction Hour"] = (
        quarter_df["Transaction Time"].dt.hour
    )


    # sales by hour

    hourly_sales = (
        quarter_df.groupby("Transaction Hour")["Revenue"]
        .sum()
    )

    sales_by_hour = (
        quarter_df.groupby("Transaction Hour")
        .agg(
            revenue=("Revenue", "sum"),
            transactions=("Transaction ID", "nunique"),
            units_sold=("Transaction Quantity", "sum")
        )
        .sort_index()
    )

    peak_hour = hourly_sales.idxmax()


    # sales by day of week

    quarter_df["Day of Week"] = (
        quarter_df["Transaction Date"].dt.day_name()
    )

    day_order = [
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ]

    sales_by_weekday = (
        quarter_df.groupby("Day of Week")["Revenue"]
        .sum()
        .reindex(day_order)
    )


    # sales by time period

    def time_period(hour):
        if 6 <= hour < 10:
            return "Morning"
        elif 11 <= hour < 13:
            return "Lunch"
        elif 14 <= hour < 16:
            return "Afternoon"
        else:
            return "Evening"


    quarter_df["Time Period"] = (
        quarter_df["Transaction Hour"].apply(time_period)
    )

    time_order = [
        "Morning",
        "Lunch",
        "Afternoon",
        "Evening"
    ]

    quarter_df["Time Period"] = pd.Categorical(
        quarter_df["Time Period"],
        categories=time_order,
        ordered=True
    )

    sales_by_time_of_day = (
        quarter_df.groupby("Time Period")["Revenue"]
        .sum()
    )


    # revenue by product category and time period

    sales_by_period_and_category = (
        quarter_df.groupby(
            ["Time Period", "Product Category"]
        )["Revenue"]
        .sum()
        .unstack(fill_value=0)
    )


    # location related metrics

    location_sales = (
        quarter_df.groupby("Store Location")["Revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    store_sales = (
        quarter_df.groupby(["Store ID", "Store Location"])
        .agg(
            revenue=("Revenue", "sum"),
            units_sold=("Transaction Quantity", "sum"),
            transactions=("Transaction ID", "nunique")
        )
        .sort_values("revenue", ascending=False)
    )


    # heatmap data

    revenue_by_day_hour = {}

    for location in quarter_df["Store Location"].unique():

        revenue_by_day_hour[location] = (
            quarter_df[
                quarter_df["Store Location"] == location
            ]
            .groupby(
                ["Day of Week", "Transaction Hour"]
            )["Revenue"]
            .sum()
            .unstack(fill_value=0)
            .reindex(day_order)
        )


    # store all results for this quarter
    quarterly_results[quarter] = {

        "total_revenue": total_revenue,
        "total_transactions": total_transactions,
        "total_units": total_units,
        "average_transaction_value": average_transaction_value,
        "average_units_per_transaction": average_units_per_transaction,

        "product_sales": product_sales,
        "category_sales": category_sales,
        "product_detail_sales": product_sales_metrics,

        "sales_by_day": sales_by_day,
        "sales_by_hour": sales_by_hour,
        "sales_by_weekday": sales_by_weekday,
        "sales_by_time_of_day": sales_by_time_of_day,
        "sales_by_period_and_category": sales_by_period_and_category,

        "location_sales": location_sales,
        "store_sales": store_sales,
        "revenue_by_day_hour": revenue_by_day_hour
    }

In [111]:
# quarterly_results[pd.Period("2023Q1")]["category_sales"]

#### Automation

In [ ]:
from matplotlib.pyplot import bar
from openpyxl.styles import Font, Alignment
import datetime
from openpyxl.chart import BarChart, Reference, LineChart, DoughnutChart, ScatterChart, Reference, Series
from openpyxl.chart.shapes import GraphicalProperties


for quarter, results in quarterly_results.items():

    wb = Workbook()

    # formatting
    header_18 = Font(name='Calibri', size=18, bold=True, color="222222")
    header_14 = Font(name='Calibri', size=14, bold=True, color="222222")

    body_bold = Font(name='Calibri', size=11, bold=True, color="222222")
    body_regular = Font(name='Calibri', size=11, bold=False, color="222222")

    center_align = Alignment(horizontal='center', vertical='center')


    # ####### first sheet - KPIs #######
    kpi_sheet = wb.active

    # name it KPIs
    kpi_sheet.title = "KPIs"

    # set columns width
    kpi_sheet.column_dimensions['A'].width = 20
    kpi_sheet.column_dimensions['B'].width = 20
    kpi_sheet.column_dimensions['C'].width = 20
    kpi_sheet.column_dimensions['D'].width = 20
    kpi_sheet.column_dimensions['E'].width = 20

    # add todays date up top
    kpi_sheet['A1'] = datetime.datetime.now().strftime("%d %B, %Y")
    kpi_sheet.row_dimensions[1].height = 40

    # set header
    kpi_sheet['A2'] = "Sales Performance Report"
    kpi_sheet['A2'].font = Font(name='Calibri', size=28, bold=True, color="222222")
    kpi_sheet.row_dimensions[2].height = 60
    kpi_sheet.merge_cells('A2:E2')
    kpi_sheet['A2'].alignment = center_align

    # set subheader
    start_month = quarter.start_time.strftime("%B")
    end_month = quarter.end_time.strftime("%B")
    year = quarter.start_time.year
    #-----------------------------------
    kpi_sheet["A3"] = f"{start_month} – {end_month} {year}"
    kpi_sheet['A3'].font = header_18
    kpi_sheet.row_dimensions[3].height = 36
    kpi_sheet.merge_cells('A3:E3')
    kpi_sheet['A3'].alignment = center_align

    # blank row
    kpi_sheet["A4"] = ""
    kpi_sheet.row_dimensions[4].height = 30

    # FIRST ROW OF KPIs
    # headers
    kpi_sheet.row_dimensions[5].height = 20
    kpi_sheet["A5"] = "TOTAL REVENUE"
    kpi_sheet["C5"] = "TRANSACTIONS"
    kpi_sheet["E5"] = "AVG TRANSACTION"
    for cell in kpi_sheet[5]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[6].height = 20
    kpi_sheet["A6"] = f"${total_revenue:,.2f}"
    kpi_sheet["C6"] = total_transactions
    kpi_sheet["E6"] = f"${average_transaction_value:,.2f}"
    for cell in kpi_sheet[6]:
        cell.alignment = center_align
        cell.font = body_regular

    # SECOND ROW OF KPIs
    # blank row
    kpi_sheet["A7"] = ""
    kpi_sheet.row_dimensions[7].height = 30

    # headers
    kpi_sheet.row_dimensions[8].height = 20
    kpi_sheet["A8"] = "BEST LOCATION"
    kpi_sheet["C8"] = "BEST PRODUCT "
    kpi_sheet["E8"] = "PEAK PERIOD"
    for cell in kpi_sheet[8]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[9].height = 20
    kpi_sheet["A9"] = store_sales.index[0][1]  # best location
    kpi_sheet["C9"] = product_sales.index[0]  # best product
    kpi_sheet["E9"] = f"{peak_hour}:00 - {peak_hour + 1}:00"  # peak period

    for cell in kpi_sheet[9]:
        cell.alignment = center_align
        cell.font = body_regular

    # ######## SECOND SHEET - SALES ANALYSIS #######
    sales_analysis = wb.create_sheet(title="Sales Analysis")

    # set columns width
    sales_analysis.column_dimensions['A'].width = 18
    sales_analysis.column_dimensions['B'].width = 18
    sales_analysis.column_dimensions['C'].width = 18
    sales_analysis.column_dimensions['D'].width = 18
    sales_analysis.column_dimensions['E'].width = 18
    sales_analysis.column_dimensions['F'].width = 18
    sales_analysis.column_dimensions['G'].width = 18
    sales_analysis.column_dimensions['H'].width = 18
    sales_analysis.column_dimensions['I'].width = 18

    # set header
    sales_analysis['A1'] = "Sales Analysis"
    sales_analysis['A1'].font = header_18
    sales_analysis.row_dimensions[1].height = 60
    sales_analysis.merge_cells('A1:I1')
    sales_analysis['A1'].alignment = center_align

    # set subheader
    sales_analysis['A2'] = "Revenue performance across products and categories"
    sales_analysis['A2'].font = header_14
    sales_analysis.row_dimensions[2].height = 36
    sales_analysis.merge_cells('A2:I2')
    sales_analysis['A2'].alignment = center_align

    # blank row
    sales_analysis["A3"] = ""
    sales_analysis.row_dimensions[3].height = 30


    #############################################

    # CHART DATA SHEET
    chart_data_ws = wb.create_sheet("Chart Data")

   
    # REVENUE BY CATEGORY
    category_data = category_sales.reset_index()

    category_data.columns = ["Product Category", "Revenue"]

    chart_data_ws["A1"] = "Product Category"
    chart_data_ws["B1"] = "Revenue"

    for row_num, row in enumerate(
        category_data.itertuples(index=False),
        start=2
        ):
        chart_data_ws.cell(row=row_num, column=1, value=row[0])
        chart_data_ws.cell(row=row_num, column=2, value=row[1])

    sales_analysis['A4'] = "Revenue by Product Category"
    sales_analysis['A4'].font = header_14
    sales_analysis.row_dimensions[4].height = 36
    
    category_chart = BarChart()

    category_chart.type = "bar"
    category_chart.style = 13

    category_chart.legend = None

    category_chart.x_axis.majorGridlines = None
    category_chart.y_axis.majorGridlines = None

    category_chart.x_axis.delete = False
    category_chart.y_axis.delete = False

    category_chart.x_axis.majorTickMark = "out"
    category_chart.y_axis.majorTickMark = "out"

    category_chart.x_axis.tickLblPos = "nextTo"
    category_chart.y_axis.tickLblPos = "nextTo"

    category_chart.y_axis.majorUnit = 25000


    category_chart.x_axis.tickLblPos = "low"
    category_chart.x_axis.numFmt = '$#,##0'

    category_data_ref = Reference(
        chart_data_ws,
        min_col=2,
        min_row=1,
        max_row=len(category_data) + 1
    )

    category_labels = Reference(
        chart_data_ws,
        min_col=1,
        min_row=2,
        max_row=len(category_data) + 1
    )

    category_chart.add_data(
        category_data_ref,
        titles_from_data=True
    )

    category_chart.set_categories(category_labels)

    category_chart.series[0].graphicalProperties.solidFill = "8FD7D7"
    category_chart.series[0].graphicalProperties.line.solidFill = "8FD7D7"
    

    category_chart.height = 10
    category_chart.width = 15

    sales_analysis.row_dimensions[5].height = category_chart.height * 28.35

    sales_analysis.add_chart(
        category_chart,
        "A5"
    )

    # REVENUE BY MONTH
    monthly_sales = results["sales_by_day"].copy()

    monthly_sales.index = pd.to_datetime(monthly_sales.index)

    monthly_sales = (
        monthly_sales
        .resample("ME")
        .sum()
    )

    monthly_data = monthly_sales.reset_index()

    monthly_data.columns = ["Month", "Revenue"]

    chart_data_ws["D1"] = "Month"
    chart_data_ws["E1"] = "Revenue"

    for row_num, row in enumerate(
        monthly_data.itertuples(index=False),
        start=2
    ):
        chart_data_ws.cell(
            row=row_num,
            column=4,
            value=row[0].strftime("%B")
        )
        chart_data_ws.cell(
            row=row_num,
            column=5,
            value=row[1]
        )

    sales_analysis['F4'] = "Revenue by Month"
    sales_analysis['F4'].font = header_14
    sales_analysis.row_dimensions[4].height = 36

    monthly_chart = LineChart()
    monthly_chart.style = 13
    monthly_chart.legend = None

    monthly_data_ref = Reference(
        chart_data_ws,
        min_col=5,
        min_row=1,
        max_row=len(monthly_data) + 1
    )

    monthly_labels = Reference(
        chart_data_ws,
        min_col=4,
        min_row=2,
        max_row=len(monthly_data) + 1
    )

    monthly_chart.add_data(
        monthly_data_ref,
        titles_from_data=True
    )

    monthly_chart.set_categories(
    monthly_labels
    )

    monthly_chart.x_axis.majorTickMark = "out"
    monthly_chart.y_axis.majorTickMark = "out"

    monthly_chart.x_axis.majorGridlines = None
    monthly_chart.y_axis.majorGridlines = None

    monthly_chart.x_axis.delete = False
    monthly_chart.y_axis.delete = False

    monthly_chart.x_axis.tickLblPos = "low"
    monthly_chart.y_axis.tickLblPos = "nextTo"

    monthly_chart.y_axis.majorUnit = 25000
    monthly_chart.y_axis.numFmt = '$#,##0'

    monthly_chart.series[0].graphicalProperties.solidFill = "EDDCA5"
    monthly_chart.series[0].graphicalProperties.line.solidFill = "EDDCA5"

    monthly_chart.height = 10
    monthly_chart.width = 15

    sales_analysis.row_dimensions[5].height = monthly_chart.height * 28.35

    sales_analysis.add_chart(
        monthly_chart,
        "F5"
    )
    # blank row
    sales_analysis["A3"] = ""
    sales_analysis.row_dimensions[3].height = 30

    # REVENUE BY TIME OF DAY
    # 00B0BE MED TEAL AND C99B38 FOR MED BROWN
    sales_analysis["A7"] = "Revenue by Time of Day"
    sales_analysis["A7"].font = header_14
    sales_analysis.row_dimensions[7].height = 36

    time_sales = results["sales_by_time_of_day"]

    # chart data
    chart_data_ws["G1"] = "Time Period"
    chart_data_ws["H1"] = "Revenue"

    for row_num, (time_period, revenue) in enumerate(
        time_sales.items(),
        start=2
    ):
        chart_data_ws.cell(
            row=row_num,
            column=7,
            value=time_period
        )

        chart_data_ws.cell(
            row=row_num,
            column=8,
            value=revenue
        )

    # create chart
    time_chart = DoughnutChart()

    time_data_ref = Reference(
        chart_data_ws,
        min_col=8,
        min_row=1,
        max_row=len(time_sales) + 1
    )

    time_labels = Reference(
        chart_data_ws,
        min_col=7,
        min_row=2,
        max_row=len(time_sales) + 1
    )

    time_chart.add_data(
        time_data_ref,
        titles_from_data=True
    )

    time_chart.set_categories(time_labels)
    time_chart.style = 13
    time_chart.height = 10
    time_chart.width = 15
    time_chart.holeSize = 45
    sales_analysis.row_dimensions[8].height = time_chart.height * 28.35

    time_chart.legend.position = "r"

    # add chart
    sales_analysis.add_chart(
        time_chart,
        "A8"
    )

    # TRANSACTIONS PER DAY VS REVENUE BY LOCATION
    sales_analysis["F7"] = "Transactions per Day vs Revenue"
    sales_analysis["F7"].font = header_14
    sales_analysis.row_dimensions[7].height = 36


    # DAILY LOCATION DATA
    daily_location_sales = (   
    quarter_df
    .groupby([
        "Store Location",
        "Transaction Date"
    ])
    .agg(
        revenue=("Revenue", "sum"),
        transactions=("Transaction ID", "nunique")
    )
    .reset_index()
    )

    # WRITE CHART DATA

    locations = [
    "Astoria",
    "Hell's Kitchen",
    "Lower Manhattan"
]

    chart_data_ws["M1"] = "Astoria Transactions"
    chart_data_ws["N1"] = "Astoria Revenue"

    chart_data_ws["P1"] = "Hell's Kitchen Transactions"
    chart_data_ws["Q1"] = "Hell's Kitchen Revenue"

    chart_data_ws["S1"] = "Lower Manhattan Transactions"
    chart_data_ws["T1"] = "Lower Manhattan Revenue"

    for col_start, location in zip(
        [13, 16, 19],
        locations
    ):

        location_data = (
            daily_location_sales[
                daily_location_sales["Store Location"] == location
            ]
            .sort_values("Transaction Date")
        )

        for row_num, row in enumerate(
            location_data.itertuples(index=False),
            start=2
        ):

            chart_data_ws.cell(
                row=row_num,
                column=col_start,
                value=row.transactions
            )

            chart_data_ws.cell(
                row=row_num,
                column=col_start + 1,
                value=row.revenue
            )

    # CREATE SCATTER CHART

    scatter_chart = ScatterChart()

    scatter_chart.style = 11

    scatter_chart.x_axis.title = "Transactions per Day"
    scatter_chart.y_axis.title = "Revenue"

    scatter_chart.x_axis.delete = False
    scatter_chart.y_axis.delete = False

    min_transactions = daily_location_sales["transactions"].min()
    max_transactions = daily_location_sales["transactions"].max()

    scatter_chart.x_axis.scaling.min = min_transactions - 10
    scatter_chart.x_axis.scaling.max = max_transactions + 10

    scatter_chart.x_axis.majorTickMark = "out"
    scatter_chart.y_axis.majorTickMark = "out"

    scatter_chart.x_axis.tickLblPos = "low"
    scatter_chart.y_axis.tickLblPos = "nextTo"

    scatter_chart.x_axis.majorGridlines = None
    scatter_chart.y_axis.majorGridlines = None

    scatter_chart.y_axis.numFmt = '$#,##0'

    scatter_chart.height = 10
    scatter_chart.width = 15


    # ADD EACH LOCATION AS A SEPARATE SERIES
    location_columns = [
        (13, 14, "Astoria"),
        (16, 17, "Hell's Kitchen"),
        (19, 20, "Lower Manhattan")
    ]

    for x_col, y_col, location in location_columns:

        location_data = (
            daily_location_sales[
                daily_location_sales["Store Location"] == location
            ]
        )

        x_values = Reference(
            chart_data_ws,
            min_col=x_col,
            min_row=2,
            max_row=len(location_data) + 1
        )

        y_values = Reference(
            chart_data_ws,
            min_col=y_col,
            min_row=2,
            max_row=len(location_data) + 1
        )

        series = Series(
            y_values,
            x_values,
            title=location
        )

        series.marker.symbol = "circle"
        series.marker.size = 3
        series.marker.graphicalProperties.shadow = None
        series.graphicalProperties.line.noFill = True
        scatter_chart.series.append(series)


    # LEGEND
    scatter_chart.legend.position = "r"

    # ADD CHART
    sales_analysis.add_chart(
        scatter_chart,
        "F8"
    )

    ###############################################################
    # HEATMAP DATA
    heatmap_start_row = 35

    sales_analysis[f"A{heatmap_start_row}"] = "Revenue by Location, Day and Hour"
    sales_analysis[f"A{heatmap_start_row}"].font = header_14

    sales_analysis.merge_cells(
        start_row=heatmap_start_row,
        start_column=1,
        end_row=heatmap_start_row,
        end_column=9
    )

    sales_analysis[f"A{heatmap_start_row}"].alignment = center_align


    heatmap_row = heatmap_start_row + 2

    for location, heatmap_data in results["revenue_by_day_hour"].items():

        sales_analysis.cell(
            row=heatmap_row,
            column=1,
            value=location
        )

        sales_analysis.cell(
            row=heatmap_row,
            column=1
        ).font = body_bold

        heatmap_row += 1

        # Header row
        sales_analysis.cell(
            row=heatmap_row,
            column=1,
            value="Day"
        )

        for col_num, hour in enumerate(
            heatmap_data.columns,
            start=2
        ):
            sales_analysis.cell(
                row=heatmap_row,
                column=col_num,
                value=f"{hour:02d}:00"
            )

        heatmap_row += 1

        # Data
        for day in heatmap_data.index:

            sales_analysis.cell(
                row=heatmap_row,
                column=1,
                value=day
            )

            for col_num, hour in enumerate(
                heatmap_data.columns,
                start=2
            ):

                value = heatmap_data.loc[day, hour]

                sales_analysis.cell(
                    row=heatmap_row,
                    column=col_num,
                    value=float(value)
                )

            heatmap_row += 1

        heatmap_row += 2


    ######## THIRD SHEET - OBSERVATIONS #######
    observations = wb.create_sheet(title="observations")

    # set columns width
    observations.column_dimensions['A'].width = 20
    observations.column_dimensions['B'].width = 20
    observations.column_dimensions['C'].width = 20
    observations.column_dimensions['D'].width = 20
    observations.column_dimensions['E'].width = 20

    # set header
    observations['A1'] = "Key Observations"
    observations['A1'].font = header_18
    observations.row_dimensions[1].height = 60
    observations.merge_cells('A1:E1')
    observations['A1'].alignment = center_align

    # set subheader
    observations['A3'] = "Automated findings highlighting significant trends and patterns with identified areas for action."
    observations['A3'].font = header_14
    observations.row_dimensions[3].height = 36
    observations.merge_cells('A3:E3')
    observations['A3'].alignment = center_align

    # blank row
    observations["A4"] = ""
    observations.row_dimensions[4].height = 30

    #-------------------------------------------
    observation_start_row = heatmap_row

    sales_analysis.cell(
        row=observation_start_row,
        column=1,
        value="Key Observations"
    )

    sales_analysis.cell(
        row=observation_start_row,
        column=1
    ).font = header_14

    sales_analysis.merge_cells(
        start_row=observation_start_row,
        start_column=1,
        end_row=observation_start_row,
        end_column=9
    )

    sales_analysis.cell(
        row=observation_start_row,
        column=1
    ).alignment = center_align




    # hide helper sheets
    chart_data_ws.sheet_state = "hidden"

    # back to first sheet
    # wb.active = kpi_sheet
    wb.active = sales_analysis

    # Save the file
    report_name = (
    f"Performance Report - "
    f"{quarter} - {start_month} – {end_month} {year}.xlsx"
    )
    #------------------------------------
    print(f"Saving report: {report_name}")
    wb.save(report_name)

Saving report: Performance Report - 2023Q1 - January – March 2023.xlsx
Saving report: Performance Report - 2023Q2 - April – June 2023.xlsx


In [198]:
from matplotlib.pyplot import bar
from openpyxl.styles import Font, Alignment
import datetime
from openpyxl.chart import BarChart, Reference, LineChart, DoughnutChart, ScatterChart, Reference, Series
from openpyxl.chart.shapes import GraphicalProperties


for quarter, results in quarterly_results.items():

    wb = Workbook()

    # formatting
    header_18 = Font(name='Calibri', size=18, bold=True, color="222222")
    header_14 = Font(name='Calibri', size=14, bold=True, color="222222")

    body_bold = Font(name='Calibri', size=11, bold=True, color="222222")
    body_regular = Font(name='Calibri', size=11, bold=False, color="222222")

    center_align = Alignment(horizontal='center', vertical='center')


    # ####### first sheet - KPIs #######
    kpi_sheet = wb.active

    # name it KPIs
    kpi_sheet.title = "KPIs"

    # set columns width
    kpi_sheet.column_dimensions['A'].width = 20
    kpi_sheet.column_dimensions['B'].width = 20
    kpi_sheet.column_dimensions['C'].width = 20
    kpi_sheet.column_dimensions['D'].width = 20
    kpi_sheet.column_dimensions['E'].width = 20

    # add todays date up top
    kpi_sheet['A1'] = datetime.datetime.now().strftime("%d %B, %Y")
    kpi_sheet.row_dimensions[1].height = 40

    # set header
    kpi_sheet['A2'] = "Sales Performance Report"
    kpi_sheet['A2'].font = Font(name='Calibri', size=28, bold=True, color="222222")
    kpi_sheet.row_dimensions[2].height = 60
    kpi_sheet.merge_cells('A2:E2')
    kpi_sheet['A2'].alignment = center_align

    # set subheader
    start_month = quarter.start_time.strftime("%B")
    end_month = quarter.end_time.strftime("%B")
    year = quarter.start_time.year
    #-----------------------------------
    kpi_sheet["A3"] = f"{start_month} – {end_month} {year}"
    kpi_sheet['A3'].font = header_18
    kpi_sheet.row_dimensions[3].height = 36
    kpi_sheet.merge_cells('A3:E3')
    kpi_sheet['A3'].alignment = center_align

    # blank row
    kpi_sheet["A4"] = ""
    kpi_sheet.row_dimensions[4].height = 30

    # FIRST ROW OF KPIs
    # headers
    kpi_sheet.row_dimensions[5].height = 20
    kpi_sheet["A5"] = "TOTAL REVENUE"
    kpi_sheet["C5"] = "TRANSACTIONS"
    kpi_sheet["E5"] = "AVG TRANSACTION"
    for cell in kpi_sheet[5]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[6].height = 20
    kpi_sheet["A6"] = f"${total_revenue:,.2f}"
    kpi_sheet["C6"] = total_transactions
    kpi_sheet["E6"] = f"${average_transaction_value:,.2f}"
    for cell in kpi_sheet[6]:
        cell.alignment = center_align
        cell.font = body_regular

    # SECOND ROW OF KPIs
    # blank row
    kpi_sheet["A7"] = ""
    kpi_sheet.row_dimensions[7].height = 30

    # headers
    kpi_sheet.row_dimensions[8].height = 20
    kpi_sheet["A8"] = "BEST LOCATION"
    kpi_sheet["C8"] = "BEST PRODUCT "
    kpi_sheet["E8"] = "PEAK PERIOD"
    for cell in kpi_sheet[8]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[9].height = 20
    kpi_sheet["A9"] = store_sales.index[0][1]  # best location
    kpi_sheet["C9"] = product_sales.index[0]  # best product
    kpi_sheet["E9"] = f"{peak_hour}:00 - {peak_hour + 1}:00"  # peak period

    for cell in kpi_sheet[9]:
        cell.alignment = center_align
        cell.font = body_regular

    # ######## SECOND SHEET - SALES ANALYSIS #######
    sales_analysis = wb.create_sheet(title="Sales Analysis")

    # set columns width
    sales_analysis.column_dimensions['A'].width = 18
    sales_analysis.column_dimensions['B'].width = 18
    sales_analysis.column_dimensions['C'].width = 18
    sales_analysis.column_dimensions['D'].width = 18
    sales_analysis.column_dimensions['E'].width = 18
    sales_analysis.column_dimensions['F'].width = 18
    sales_analysis.column_dimensions['G'].width = 18
    sales_analysis.column_dimensions['H'].width = 18
    sales_analysis.column_dimensions['I'].width = 18

    # set header
    sales_analysis['A1'] = "Sales Analysis"
    sales_analysis['A1'].font = header_18
    sales_analysis.row_dimensions[1].height = 60
    sales_analysis.merge_cells('A1:I1')
    sales_analysis['A1'].alignment = center_align

    # set subheader
    sales_analysis['A2'] = "Revenue performance across products and categories"
    sales_analysis['A2'].font = header_14
    sales_analysis.row_dimensions[2].height = 36
    sales_analysis.merge_cells('A2:I2')
    sales_analysis['A2'].alignment = center_align

    # blank row
    sales_analysis["A3"] = ""
    sales_analysis.row_dimensions[3].height = 30

    # CHART DATA SHEET
    chart_data_ws = wb.create_sheet("Chart Data")

    # REVENUE BY CATEGORY
    category_data = category_sales.reset_index()

    category_data.columns = ["Product Category", "Revenue"]

    chart_data_ws["A1"] = "Product Category"
    chart_data_ws["B1"] = "Revenue"

    for row_num, row in enumerate(
        category_data.itertuples(index=False),
        start=2
        ):
        chart_data_ws.cell(row=row_num, column=1, value=row[0])
        chart_data_ws.cell(row=row_num, column=2, value=row[1])

    sales_analysis['A4'] = "Revenue by Product Category"
    sales_analysis['A4'].font = header_14
    sales_analysis.row_dimensions[4].height = 36
    
    category_chart = BarChart()

    category_chart.type = "bar"
    category_chart.style = 13

    category_chart.legend = None

    category_chart.x_axis.majorGridlines = None
    category_chart.y_axis.majorGridlines = None

    category_chart.x_axis.delete = False
    category_chart.y_axis.delete = False

    category_chart.x_axis.majorTickMark = "out"
    category_chart.y_axis.majorTickMark = "out"

    category_chart.x_axis.tickLblPos = "nextTo"
    category_chart.y_axis.tickLblPos = "nextTo"

    category_chart.y_axis.majorUnit = 25000

    category_chart.x_axis.tickLblPos = "low"
    category_chart.x_axis.numFmt = '$#,##0'

    category_data_ref = Reference(
        chart_data_ws,
        min_col=2,
        min_row=1,
        max_row=len(category_data) + 1
    )

    category_labels = Reference(
        chart_data_ws,
        min_col=1,
        min_row=2,
        max_row=len(category_data) + 1
    )

    category_chart.add_data(
        category_data_ref,
        titles_from_data=True
    )

    category_chart.set_categories(category_labels)
    
    category_chart.height = 10
    category_chart.width = 15

    sales_analysis.row_dimensions[5].height = category_chart.height * 28.35

    sales_analysis.add_chart(
        category_chart,
        "A5"
    )

    # REVENUE BY MONTH
    monthly_sales = results["sales_by_day"].copy()

    monthly_sales.index = pd.to_datetime(monthly_sales.index)

    monthly_sales = (
        monthly_sales
        .resample("ME")
        .sum()
    )

    monthly_data = monthly_sales.reset_index()

    monthly_data.columns = ["Month", "Revenue"]

    chart_data_ws["D1"] = "Month"
    chart_data_ws["E1"] = "Revenue"

    for row_num, row in enumerate(
        monthly_data.itertuples(index=False),
        start=2
    ):
        chart_data_ws.cell(
            row=row_num,
            column=4,
            value=row[0].strftime("%B")
        )
        chart_data_ws.cell(
            row=row_num,
            column=5,
            value=row[1]
        )

    sales_analysis['F4'] = "Revenue by Month"
    sales_analysis['F4'].font = header_14
    sales_analysis.row_dimensions[4].height = 36

    monthly_chart = LineChart()
    monthly_chart.style = 13
    monthly_chart.legend = None

    monthly_data_ref = Reference(
        chart_data_ws,
        min_col=5,
        min_row=1,
        max_row=len(monthly_data) + 1
    )

    monthly_labels = Reference(
        chart_data_ws,
        min_col=4,
        min_row=2,
        max_row=len(monthly_data) + 1
    )

    monthly_chart.add_data(
        monthly_data_ref,
        titles_from_data=True
    )

    monthly_chart.set_categories(
    monthly_labels
    )

    monthly_chart.x_axis.majorTickMark = "out"
    monthly_chart.y_axis.majorTickMark = "out"

    monthly_chart.x_axis.majorGridlines = None
    monthly_chart.y_axis.majorGridlines = None

    monthly_chart.x_axis.delete = False
    monthly_chart.y_axis.delete = False

    monthly_chart.x_axis.tickLblPos = "low"
    monthly_chart.y_axis.tickLblPos = "nextTo"

    monthly_chart.y_axis.majorUnit = 25000
    monthly_chart.y_axis.numFmt = '$#,##0'

    # monthly_chart.series[0].graphicalProperties.solidFill = "EDDCA5"
    # monthly_chart.series[0].graphicalProperties.line.solidFill = "EDDCA5"

    monthly_chart.height = 10
    monthly_chart.width = 15

    sales_analysis.row_dimensions[5].height = monthly_chart.height * 28.35

    sales_analysis.add_chart(
        monthly_chart,
        "F5"
    )
    # blank row
    sales_analysis["A3"] = ""
    sales_analysis.row_dimensions[3].height = 30

    # REVENUE BY TIME OF DAY
    # 00B0BE MED TEAL AND C99B38 FOR MED BROWN
    sales_analysis["A7"] = "Revenue by Time of Day"
    sales_analysis["A7"].font = header_14
    sales_analysis.row_dimensions[7].height = 36

    time_sales = results["sales_by_time_of_day"]

    # chart data
    chart_data_ws["G1"] = "Time Period"
    chart_data_ws["H1"] = "Revenue"

    for row_num, (time_period, revenue) in enumerate(
        time_sales.items(),
        start=2
    ):
        chart_data_ws.cell(
            row=row_num,
            column=7,
            value=time_period
        )

        chart_data_ws.cell(
            row=row_num,
            column=8,
            value=revenue
        )

    # create chart
    time_chart = DoughnutChart()

    time_data_ref = Reference(
        chart_data_ws,
        min_col=8,
        min_row=1,
        max_row=len(time_sales) + 1
    )

    time_labels = Reference(
        chart_data_ws,
        min_col=7,
        min_row=2,
        max_row=len(time_sales) + 1
    )

    time_chart.add_data(
        time_data_ref,
        titles_from_data=True
    )

    time_chart.set_categories(time_labels)
    time_chart.style = 13

    time_chart.height = 10
    time_chart.width = 15
    time_chart.holeSize = 45
    sales_analysis.row_dimensions[8].height = time_chart.height * 28.35

    time_chart.legend.position = "r"

    # add chart
    sales_analysis.add_chart(
        time_chart,
        "A8"
    )

    # TRANSACTIONS PER DAY VS REVENUE

    sales_analysis["F7"] = "Transactions per Day vs Revenue"
    sales_analysis["F7"].font = header_14
    sales_analysis.row_dimensions[7].height = 36

    # DAILY LOCATION DATA

    daily_location_sales = (
        quarter_df
        .groupby([
            "Store Location",
            "Transaction Date"
        ])
        .agg(
            revenue=("Revenue", "sum"),
            transactions=("Transaction ID", "nunique")
        )
        .reset_index()
    )

    # WRITE CHART DATA

    locations = [
        "Astoria",
        "Hell's Kitchen",
        "Lower Manhattan"
    ]

    chart_data_ws["M1"] = "Astoria Transactions"
    chart_data_ws["N1"] = "Astoria Revenue"

    chart_data_ws["P1"] = "Hell's Kitchen Transactions"
    chart_data_ws["Q1"] = "Hell's Kitchen Revenue"

    chart_data_ws["S1"] = "Lower Manhattan Transactions"
    chart_data_ws["T1"] = "Lower Manhattan Revenue"


    for col_start, location in zip(
        [13, 16, 19],
        locations
    ):

        location_data = (
            daily_location_sales[
                daily_location_sales["Store Location"] == location
            ]
            .sort_values("Transaction Date")
        )

        for row_num, row in enumerate(
            location_data.itertuples(index=False),
            start=2
        ):

            chart_data_ws.cell(
                row=row_num,
                column=col_start,
                value=row.transactions
            )

            chart_data_ws.cell(
                row=row_num,
                column=col_start + 1,
                value=row.revenue
            )

    # CREATE SCATTER CHART
    scatter_chart = ScatterChart()

    scatter_chart.style = 13

    # NO AXIS TITLES
    scatter_chart.x_axis.title = None
    scatter_chart.y_axis.title = None

    scatter_chart.x_axis.delete = False
    scatter_chart.y_axis.delete = False

    # X-AXIS RANGE

    min_transactions = daily_location_sales["transactions"].min()
    max_transactions = daily_location_sales["transactions"].max()

    scatter_chart.x_axis.scaling.min = min_transactions - 10
    scatter_chart.x_axis.scaling.max = max_transactions + 10

    # AXIS TICKS
    scatter_chart.x_axis.majorTickMark = "out"
    scatter_chart.y_axis.majorTickMark = "out"

    scatter_chart.x_axis.tickLblPos = "low"
    scatter_chart.y_axis.tickLblPos = "nextTo"

    # NO GRIDLINES
    scatter_chart.x_axis.majorGridlines = None
    scatter_chart.y_axis.majorGridlines = None

    # REVENUE FORMAT
    scatter_chart.y_axis.numFmt = '$#,##0'

    scatter_chart.height = 10
    scatter_chart.width = 15

    # ADD EACH LOCATION AS A SEPARATE SERIES

    location_columns = [
        (13, 14, "Astoria"),
        (16, 17, "Hell's Kitchen"),
        (19, 20, "Lower Manhattan")
    ]

    for x_col, y_col, location in location_columns:

        location_data = (
            daily_location_sales[
                daily_location_sales["Store Location"] == location
            ]
        )

        x_values = Reference(
            chart_data_ws,
            min_col=x_col,
            min_row=2,
            max_row=len(location_data) + 1
        )

        y_values = Reference(
            chart_data_ws,
            min_col=y_col,
            min_row=2,
            max_row=len(location_data) + 1
        )

        series = Series(
            y_values,
            x_values,
            title=location
        )

        # DOTS ONLY

        series.marker.symbol = "circle"
        series.marker.graphicalProperties.shadow = None

        # NO CONNECTING LINES

        series.graphicalProperties.line.noFill = True

        scatter_chart.series.append(series)

    # LEGEND
    scatter_chart.legend.position = "r"

    # ADD CHART

    sales_analysis.add_chart(
        scatter_chart,
        "F8"
    )

    # HEATMAPS — REVENUE BY LOCATION, DAY AND HOUR

    # VISIBLE SHEET HEADING
    heatmap_start_row = 10

    sales_analysis[f"A{heatmap_start_row}"] = (
        "Revenue by Location, Day and Hour"
    )

    sales_analysis[f"A{heatmap_start_row}"].font = header_14

    sales_analysis.row_dimensions[
        heatmap_start_row
    ].height = 36

    sales_analysis.merge_cells(
        start_row=heatmap_start_row,
        start_column=1,
        end_row=heatmap_start_row,
        end_column=9
    )

    sales_analysis[
        f"A{heatmap_start_row}"
    ].alignment = center_align

    # HEATMAP DATA
    heatmap_start_col = 22  # Column V

    heatmap_row = 1

    heatmap_style = 11


    for location, heatmap_data in results[
        "revenue_by_day_hour"
    ].items():

        # WRITE LOCATION NAME TO HIDDEN SHEET

        chart_data_ws.cell(
            row=heatmap_row,
            column=heatmap_start_col,
            value=location
        )

        heatmap_row += 1

        # WRITE HEADER ROW
        chart_data_ws.cell(
            row=heatmap_row,
            column=heatmap_start_col,
            value="Day"
        )

        for col_num, hour in enumerate(
            heatmap_data.columns,
            start=heatmap_start_col + 1
        ):

            chart_data_ws.cell(
                row=heatmap_row,
                column=col_num,
                value=f"{hour:02d}:00"
            )

        heatmap_row += 1


        # WRITE HEATMAP DATA
        for day in heatmap_data.index:

            chart_data_ws.cell(
                row=heatmap_row,
                column=heatmap_start_col,
                value=day
            )

            for col_num, hour in enumerate(
                heatmap_data.columns,
                start=heatmap_start_col + 1
            ):

                value = heatmap_data.loc[day, hour]

                chart_data_ws.cell(
                    row=heatmap_row,
                    column=col_num,
                    value=float(value)
                )

            heatmap_row += 1


        # CREATE HEATMAP 
        heatmap_chart = BarChart()

        heatmap_chart.style = heatmap_style

        heatmap_chart.legend = None

        heatmap_chart.height = 8
        heatmap_chart.width = 12

        # NEXT STYLE
        heatmap_style += 1

        # hide helper sheets
        chart_data_ws.sheet_state = "hidden"

        ######## THIRD SHEET - OBSERVATIONS #######
        observations = wb.create_sheet(title="observations")

        # set columns width
        observations.column_dimensions['A'].width = 20
        observations.column_dimensions['B'].width = 20
        observations.column_dimensions['C'].width = 20
        observations.column_dimensions['D'].width = 20
        observations.column_dimensions['E'].width = 20

        # set header
        observations['A1'] = "Key Observations"
        observations['A1'].font = header_18
        observations.row_dimensions[1].height = 60
        observations.merge_cells('A1:E1')
        observations['A1'].alignment = center_align

        # set subheader
        observations['A3'] = "Automated findings highlighting significant trends and patterns with identified areas for action."
        observations['A3'].font = header_14
        observations.row_dimensions[3].height = 36
        observations.merge_cells('A3:E3')
        observations['A3'].alignment = center_align

        # blank row
        observations["A4"] = ""
        observations.row_dimensions[4].height = 30

        #-------------------------------------------
        observation_start_row = heatmap_row

        sales_analysis.cell(
            row=observation_start_row,
            column=1,
            value="Key Observations"
        )

        sales_analysis.cell(
            row=observation_start_row,
            column=1
        ).font = header_14

        sales_analysis.merge_cells(
            start_row=observation_start_row,
            start_column=1,
            end_row=observation_start_row,
            end_column=9
        )

        sales_analysis.cell(
            row=observation_start_row,
            column=1
        ).alignment = center_align


    #################################################

    # back to first sheet
    # wb.active = kpi_sheet
    wb.active = sales_analysis

    # Save the file
    report_name = (
    f"Performance Report - "
    f"{quarter} - {start_month} – {end_month} {year}.xlsx"
    )
    #------------------------------------
    print(f"Saving report: {report_name}")
    wb.save(report_name)


Saving report: Performance Report - 2023Q1 - January – March 2023.xlsx
Saving report: Performance Report - 2023Q2 - April – June 2023.xlsx


In [ ]:
from matplotlib.pyplot import bar
from openpyxl.styles import Font, Alignment
import datetime
from openpyxl.chart import BarChart, Reference, LineChart, DoughnutChart, ScatterChart, Reference, Series
from openpyxl.chart.shapes import GraphicalProperties


for quarter, results in quarterly_results.items():

    wb = Workbook()

    # formatting
    header_18 = Font(name='Calibri', size=18, bold=True, color="222222")
    header_14 = Font(name='Calibri', size=14, bold=True, color="222222")

    body_bold = Font(name='Calibri', size=11, bold=True, color="222222")
    body_regular = Font(name='Calibri', size=11, bold=False, color="222222")

    center_align = Alignment(horizontal='center', vertical='center')


    # ####### first sheet - KPIs #######
    kpi_sheet = wb.active

    # name it KPIs
    kpi_sheet.title = "KPIs"

    # set columns width
    kpi_sheet.column_dimensions['A'].width = 20
    kpi_sheet.column_dimensions['B'].width = 20
    kpi_sheet.column_dimensions['C'].width = 20
    kpi_sheet.column_dimensions['D'].width = 20
    kpi_sheet.column_dimensions['E'].width = 20

    # add todays date up top
    kpi_sheet['A1'] = datetime.datetime.now().strftime("%d %B, %Y")
    kpi_sheet.row_dimensions[1].height = 40

    # set header
    kpi_sheet['A2'] = "Sales Performance Report"
    kpi_sheet['A2'].font = Font(name='Calibri', size=28, bold=True, color="222222")
    kpi_sheet.row_dimensions[2].height = 60
    kpi_sheet.merge_cells('A2:E2')
    kpi_sheet['A2'].alignment = center_align

    # set subheader
    start_month = quarter.start_time.strftime("%B")
    end_month = quarter.end_time.strftime("%B")
    year = quarter.start_time.year
    #-----------------------------------
    kpi_sheet["A3"] = f"{start_month} – {end_month} {year}"
    kpi_sheet['A3'].font = header_18
    kpi_sheet.row_dimensions[3].height = 36
    kpi_sheet.merge_cells('A3:E3')
    kpi_sheet['A3'].alignment = center_align

    # blank row
    kpi_sheet["A4"] = ""
    kpi_sheet.row_dimensions[4].height = 30

    # FIRST ROW OF KPIs
    # headers
    kpi_sheet.row_dimensions[5].height = 20
    kpi_sheet["A5"] = "TOTAL REVENUE"
    kpi_sheet["C5"] = "TRANSACTIONS"
    kpi_sheet["E5"] = "AVG TRANSACTION"
    for cell in kpi_sheet[5]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[6].height = 20
    kpi_sheet["A6"] = f"${total_revenue:,.2f}"
    kpi_sheet["C6"] = total_transactions
    kpi_sheet["E6"] = f"${average_transaction_value:,.2f}"
    for cell in kpi_sheet[6]:
        cell.alignment = center_align
        cell.font = body_regular

    # SECOND ROW OF KPIs
    # blank row
    kpi_sheet["A7"] = ""
    kpi_sheet.row_dimensions[7].height = 30

    # headers
    kpi_sheet.row_dimensions[8].height = 20
    kpi_sheet["A8"] = "BEST LOCATION"
    kpi_sheet["C8"] = "BEST PRODUCT "
    kpi_sheet["E8"] = "PEAK PERIOD"
    for cell in kpi_sheet[8]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[9].height = 20
    kpi_sheet["A9"] = store_sales.index[0][1]  # best location
    kpi_sheet["C9"] = product_sales.index[0]  # best product
    kpi_sheet["E9"] = f"{peak_hour}:00 - {peak_hour + 1}:00"  # peak period

    for cell in kpi_sheet[9]:
        cell.alignment = center_align
        cell.font = body_regular

    # ######## SECOND SHEET - SALES ANALYSIS #######
    sales_analysis = wb.create_sheet(title="Sales Analysis")

    # set columns width
    sales_analysis.column_dimensions['A'].width = 18
    sales_analysis.column_dimensions['B'].width = 18
    sales_analysis.column_dimensions['C'].width = 18
    sales_analysis.column_dimensions['D'].width = 18
    sales_analysis.column_dimensions['E'].width = 18
    sales_analysis.column_dimensions['F'].width = 18
    sales_analysis.column_dimensions['G'].width = 18
    sales_analysis.column_dimensions['H'].width = 18
    sales_analysis.column_dimensions['I'].width = 18

    # set header
    sales_analysis['A1'] = "Sales Analysis"
    sales_analysis['A1'].font = header_18
    sales_analysis.row_dimensions[1].height = 60
    sales_analysis.merge_cells('A1:I1')
    sales_analysis['A1'].alignment = center_align

    # set subheader
    sales_analysis['A2'] = "Revenue performance across products and categories"
    sales_analysis['A2'].font = header_14
    sales_analysis.row_dimensions[2].height = 36
    sales_analysis.merge_cells('A2:I2')
    sales_analysis['A2'].alignment = center_align

    # blank row
    sales_analysis["A3"] = ""
    sales_analysis.row_dimensions[3].height = 30


    #############################################

    # CHART DATA SHEET
    chart_data_ws = wb.create_sheet("Chart Data")

   
    # REVENUE BY CATEGORY
    category_data = category_sales.reset_index()

    category_data.columns = ["Product Category", "Revenue"]

    chart_data_ws["A1"] = "Product Category"
    chart_data_ws["B1"] = "Revenue"

    for row_num, row in enumerate(
        category_data.itertuples(index=False),
        start=2
        ):
        chart_data_ws.cell(row=row_num, column=1, value=row[0])
        chart_data_ws.cell(row=row_num, column=2, value=row[1])

    sales_analysis['A4'] = "Revenue by Product Category"
    sales_analysis['A4'].font = header_14
    sales_analysis.row_dimensions[4].height = 36
    
    category_chart = BarChart()

    category_chart.type = "bar"
    category_chart.style = 13

    category_chart.legend = None

    category_chart.x_axis.majorGridlines = None
    category_chart.y_axis.majorGridlines = None

    category_chart.x_axis.delete = False
    category_chart.y_axis.delete = False

    category_chart.x_axis.majorTickMark = "out"
    category_chart.y_axis.majorTickMark = "out"

    category_chart.x_axis.tickLblPos = "nextTo"
    category_chart.y_axis.tickLblPos = "nextTo"

    category_chart.y_axis.majorUnit = 25000


    category_chart.x_axis.tickLblPos = "low"
    category_chart.x_axis.numFmt = '$#,##0'

    category_data_ref = Reference(
        chart_data_ws,
        min_col=2,
        min_row=1,
        max_row=len(category_data) + 1
    )

    category_labels = Reference(
        chart_data_ws,
        min_col=1,
        min_row=2,
        max_row=len(category_data) + 1
    )

    category_chart.add_data(
        category_data_ref,
        titles_from_data=True
    )

    category_chart.set_categories(category_labels)

    category_chart.series[0].graphicalProperties.solidFill = "8FD7D7"
    category_chart.series[0].graphicalProperties.line.solidFill = "8FD7D7"
    

    category_chart.height = 10
    category_chart.width = 15

    sales_analysis.row_dimensions[5].height = category_chart.height * 28.35

    sales_analysis.add_chart(
        category_chart,
        "A5"
    )

    # REVENUE BY MONTH
    monthly_sales = results["sales_by_day"].copy()

    monthly_sales.index = pd.to_datetime(monthly_sales.index)

    monthly_sales = (
        monthly_sales
        .resample("ME")
        .sum()
    )

    monthly_data = monthly_sales.reset_index()

    monthly_data.columns = ["Month", "Revenue"]

    chart_data_ws["D1"] = "Month"
    chart_data_ws["E1"] = "Revenue"

    for row_num, row in enumerate(
        monthly_data.itertuples(index=False),
        start=2
    ):
        chart_data_ws.cell(
            row=row_num,
            column=4,
            value=row[0].strftime("%B")
        )
        chart_data_ws.cell(
            row=row_num,
            column=5,
            value=row[1]
        )

    sales_analysis['F4'] = "Revenue by Month"
    sales_analysis['F4'].font = header_14
    sales_analysis.row_dimensions[4].height = 36

    monthly_chart = LineChart()
    monthly_chart.style = 13
    monthly_chart.legend = None

    monthly_data_ref = Reference(
        chart_data_ws,
        min_col=5,
        min_row=1,
        max_row=len(monthly_data) + 1
    )

    monthly_labels = Reference(
        chart_data_ws,
        min_col=4,
        min_row=2,
        max_row=len(monthly_data) + 1
    )

    monthly_chart.add_data(
        monthly_data_ref,
        titles_from_data=True
    )

    monthly_chart.set_categories(
    monthly_labels
    )

    monthly_chart.x_axis.majorTickMark = "out"
    monthly_chart.y_axis.majorTickMark = "out"

    monthly_chart.x_axis.majorGridlines = None
    monthly_chart.y_axis.majorGridlines = None

    monthly_chart.x_axis.delete = False
    monthly_chart.y_axis.delete = False

    monthly_chart.x_axis.tickLblPos = "low"
    monthly_chart.y_axis.tickLblPos = "nextTo"

    monthly_chart.y_axis.majorUnit = 25000
    monthly_chart.y_axis.numFmt = '$#,##0'

    monthly_chart.series[0].graphicalProperties.solidFill = "EDDCA5"
    monthly_chart.series[0].graphicalProperties.line.solidFill = "EDDCA5"

    monthly_chart.height = 10
    monthly_chart.width = 15

    sales_analysis.row_dimensions[5].height = monthly_chart.height * 28.35

    sales_analysis.add_chart(
        monthly_chart,
        "F5"
    )
    # blank row
    sales_analysis["A3"] = ""
    sales_analysis.row_dimensions[3].height = 30

    # REVENUE BY TIME OF DAY
    # 00B0BE MED TEAL AND C99B38 FOR MED BROWN
    sales_analysis["A7"] = "Revenue by Time of Day"
    sales_analysis["A7"].font = header_14
    sales_analysis.row_dimensions[7].height = 36

    time_sales = results["sales_by_time_of_day"]

    # chart data
    chart_data_ws["G1"] = "Time Period"
    chart_data_ws["H1"] = "Revenue"

    for row_num, (time_period, revenue) in enumerate(
        time_sales.items(),
        start=2
    ):
        chart_data_ws.cell(
            row=row_num,
            column=7,
            value=time_period
        )

        chart_data_ws.cell(
            row=row_num,
            column=8,
            value=revenue
        )

    # create chart
    time_chart = DoughnutChart()

    time_data_ref = Reference(
        chart_data_ws,
        min_col=8,
        min_row=1,
        max_row=len(time_sales) + 1
    )

    time_labels = Reference(
        chart_data_ws,
        min_col=7,
        min_row=2,
        max_row=len(time_sales) + 1
    )

    time_chart.add_data(
        time_data_ref,
        titles_from_data=True
    )

    time_chart.set_categories(time_labels)

    time_chart.height = 10
    time_chart.width = 15
    time_chart.holeSize = 45
    sales_analysis.row_dimensions[8].height = time_chart.height * 28.35

    time_chart.legend.position = "r"

    # add chart
    sales_analysis.add_chart(
        time_chart,
        "A8"
    )

    # TRANSACTIONS PER DAY VS REVENUE BY LOCATION

    # TRANSACTIONS PER DAY VS REVENUE

    sales_analysis["F7"] = "Transactions per Day vs Revenue"
    sales_analysis["F7"].font = header_14
    sales_analysis.row_dimensions[7].height = 36


    # DAILY LOCATION DATA

    daily_location_sales = (
        quarter_df
        .groupby([
            "Store Location",
            "Transaction Date"
        ])
        .agg(
            revenue=("Revenue", "sum"),
            transactions=("Transaction ID", "nunique")
        )
        .reset_index()
    )


    # WRITE CHART DATA

    locations = [
        "Astoria",
        "Hell's Kitchen",
        "Lower Manhattan"
    ]

    chart_data_ws["M1"] = "Astoria Transactions"
    chart_data_ws["N1"] = "Astoria Revenue"

    chart_data_ws["P1"] = "Hell's Kitchen Transactions"
    chart_data_ws["Q1"] = "Hell's Kitchen Revenue"

    chart_data_ws["S1"] = "Lower Manhattan Transactions"
    chart_data_ws["T1"] = "Lower Manhattan Revenue"


    for col_start, location in zip(
        [13, 16, 19],
        locations
    ):

        location_data = (
            daily_location_sales[
                daily_location_sales["Store Location"] == location
            ]
            .sort_values("Transaction Date")
        )

        for row_num, row in enumerate(
            location_data.itertuples(index=False),
            start=2
        ):

            chart_data_ws.cell(
                row=row_num,
                column=col_start,
                value=row.transactions
            )

            chart_data_ws.cell(
                row=row_num,
                column=col_start + 1,
                value=row.revenue
            )


    # CREATE SCATTER CHART

    scatter_chart = ScatterChart()

    scatter_chart.style = 13


    # NO AXIS TITLES

    scatter_chart.x_axis.title = None
    scatter_chart.y_axis.title = None


    scatter_chart.x_axis.delete = False
    scatter_chart.y_axis.delete = False


    # X-AXIS RANGE

    min_transactions = daily_location_sales["transactions"].min()
    max_transactions = daily_location_sales["transactions"].max()

    scatter_chart.x_axis.scaling.min = min_transactions - 10
    scatter_chart.x_axis.scaling.max = max_transactions + 10


    # AXIS TICKS

    scatter_chart.x_axis.majorTickMark = "out"
    scatter_chart.y_axis.majorTickMark = "out"

    scatter_chart.x_axis.tickLblPos = "low"
    scatter_chart.y_axis.tickLblPos = "nextTo"


    # NO GRIDLINES

    scatter_chart.x_axis.majorGridlines = None
    scatter_chart.y_axis.majorGridlines = None


    # REVENUE FORMAT

    scatter_chart.y_axis.numFmt = '$#,##0'


    scatter_chart.height = 10
    scatter_chart.width = 15


    # ADD EACH LOCATION AS A SEPARATE SERIES

    location_columns = [
        (13, 14, "Astoria"),
        (16, 17, "Hell's Kitchen"),
        (19, 20, "Lower Manhattan")
    ]


    for x_col, y_col, location in location_columns:

        location_data = (
            daily_location_sales[
                daily_location_sales["Store Location"] == location
            ]
        )

        x_values = Reference(
            chart_data_ws,
            min_col=x_col,
            min_row=2,
            max_row=len(location_data) + 1
        )

        y_values = Reference(
            chart_data_ws,
            min_col=y_col,
            min_row=2,
            max_row=len(location_data) + 1
        )

        series = Series(
            y_values,
            x_values,
            title=location
        )

        # DOTS ONLY

        series.marker.symbol = "circle"
        series.marker.graphicalProperties.shadow = None

        # NO CONNECTING LINES

        series.graphicalProperties.line.noFill = True

        scatter_chart.series.append(series)


    # LEGEND

    scatter_chart.legend.position = "r"


    # ADD CHART

    sales_analysis.add_chart(
        scatter_chart,
        "F8"
    )


    # HEATMAP DATA
    heatmap_start_row = 35

    sales_analysis[f"A{heatmap_start_row}"] = "Revenue by Location, Day and Hour"
    sales_analysis[f"A{heatmap_start_row}"].font = header_14

    sales_analysis.merge_cells(
        start_row=heatmap_start_row,
        start_column=1,
        end_row=heatmap_start_row,
        end_column=9
    )

    sales_analysis[f"A{heatmap_start_row}"].alignment = center_align


    heatmap_row = heatmap_start_row + 2

    for location, heatmap_data in results["revenue_by_day_hour"].items():

        sales_analysis.cell(
            row=heatmap_row,
            column=1,
            value=location
        )

        sales_analysis.cell(
            row=heatmap_row,
            column=1
        ).font = body_bold

        heatmap_row += 1

        # Header row
        sales_analysis.cell(
            row=heatmap_row,
            column=1,
            value="Day"
        )

        for col_num, hour in enumerate(
            heatmap_data.columns,
            start=2
        ):
            sales_analysis.cell(
                row=heatmap_row,
                column=col_num,
                value=f"{hour:02d}:00"
            )

        heatmap_row += 1

        # Data
        for day in heatmap_data.index:

            sales_analysis.cell(
                row=heatmap_row,
                column=1,
                value=day
            )

            for col_num, hour in enumerate(
                heatmap_data.columns,
                start=2
            ):

                value = heatmap_data.loc[day, hour]

                sales_analysis.cell(
                    row=heatmap_row,
                    column=col_num,
                    value=float(value)
                )

            heatmap_row += 1

        heatmap_row += 2


    ######## THIRD SHEET - OBSERVATIONS #######
    observations = wb.create_sheet(title="observations")

    # set columns width
    observations.column_dimensions['A'].width = 20
    observations.column_dimensions['B'].width = 20
    observations.column_dimensions['C'].width = 20
    observations.column_dimensions['D'].width = 20
    observations.column_dimensions['E'].width = 20

    # set header
    observations['A1'] = "Key Observations"
    observations['A1'].font = header_18
    observations.row_dimensions[1].height = 60
    observations.merge_cells('A1:E1')
    observations['A1'].alignment = center_align

    # set subheader
    observations['A3'] = "Automated findings highlighting significant trends and patterns with identified areas for action."
    observations['A3'].font = header_14
    observations.row_dimensions[3].height = 36
    observations.merge_cells('A3:E3')
    observations['A3'].alignment = center_align

    # blank row
    observations["A4"] = ""
    observations.row_dimensions[4].height = 30

    #-------------------------------------------
    observation_start_row = heatmap_row

    sales_analysis.cell(
        row=observation_start_row,
        column=1,
        value="Key Observations"
    )

    sales_analysis.cell(
        row=observation_start_row,
        column=1
    ).font = header_14

    sales_analysis.merge_cells(
        start_row=observation_start_row,
        start_column=1,
        end_row=observation_start_row,
        end_column=9
    )

    sales_analysis.cell(
        row=observation_start_row,
        column=1
    ).alignment = center_align



    ############################################

    # Rank	Product	Revenue
    # 1	    ...	    $...
    # 2	    ...	    $...
    # 3	    ...	    $...
    
    # TOP LOCATION
    # Lower Manhattan

    # REVENUE
    # $XXX,XXX

    # SHARE OF TOTAL
    # XX.X%
    # '''


    # hide helper sheets
    chart_data_ws.sheet_state = "hidden"

    # back to first sheet
    # wb.active = kpi_sheet
    wb.active = sales_analysis

    # Save the file
    report_name = (
    f"Performance Report - "
    f"{quarter} - {start_month} – {end_month} {year}.xlsx"
    )
    #------------------------------------
    print(f"Saving report: {report_name}")
    wb.save(report_name)

Saving report: Performance Report - 2023Q1 - January – March 2023.xlsx
Saving report: Performance Report - 2023Q2 - April – June 2023.xlsx


In [ ]:
from matplotlib.pyplot import bar
from openpyxl.styles import Font, Alignment
import datetime
from openpyxl.chart import BarChart, Reference, LineChart, DoughnutChart, ScatterChart, Reference, Series
from openpyxl.chart.shapes import GraphicalProperties


for quarter, results in quarterly_results.items():

    wb = Workbook()

    # formatting
    header_18 = Font(name='Calibri', size=18, bold=True, color="222222")
    header_14 = Font(name='Calibri', size=14, bold=True, color="222222")

    body_bold = Font(name='Calibri', size=11, bold=True, color="222222")
    body_regular = Font(name='Calibri', size=11, bold=False, color="222222")

    center_align = Alignment(horizontal='center', vertical='center')


    # ####### first sheet - KPIs #######
    kpi_sheet = wb.active

    # name it KPIs
    kpi_sheet.title = "KPIs"

    # set columns width
    kpi_sheet.column_dimensions['A'].width = 20
    kpi_sheet.column_dimensions['B'].width = 20
    kpi_sheet.column_dimensions['C'].width = 20
    kpi_sheet.column_dimensions['D'].width = 20
    kpi_sheet.column_dimensions['E'].width = 20

    # add todays date up top
    kpi_sheet['A1'] = datetime.datetime.now().strftime("%d %B, %Y")
    kpi_sheet.row_dimensions[1].height = 40

    # set header
    kpi_sheet['A2'] = "Sales Performance Report"
    kpi_sheet['A2'].font = Font(name='Calibri', size=28, bold=True, color="222222")
    kpi_sheet.row_dimensions[2].height = 60
    kpi_sheet.merge_cells('A2:E2')
    kpi_sheet['A2'].alignment = center_align

    # set subheader
    start_month = quarter.start_time.strftime("%B")
    end_month = quarter.end_time.strftime("%B")
    year = quarter.start_time.year
    #-----------------------------------
    kpi_sheet["A3"] = f"{start_month} – {end_month} {year}"
    kpi_sheet['A3'].font = header_18
    kpi_sheet.row_dimensions[3].height = 36
    kpi_sheet.merge_cells('A3:E3')
    kpi_sheet['A3'].alignment = center_align

    # blank row
    kpi_sheet["A4"] = ""
    kpi_sheet.row_dimensions[4].height = 30

    # FIRST ROW OF KPIs
    # headers
    kpi_sheet.row_dimensions[5].height = 20
    kpi_sheet["A5"] = "TOTAL REVENUE"
    kpi_sheet["C5"] = "TRANSACTIONS"
    kpi_sheet["E5"] = "AVG TRANSACTION"
    for cell in kpi_sheet[5]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[6].height = 20
    kpi_sheet["A6"] = f"${total_revenue:,.2f}"
    kpi_sheet["C6"] = total_transactions
    kpi_sheet["E6"] = f"${average_transaction_value:,.2f}"
    for cell in kpi_sheet[6]:
        cell.alignment = center_align
        cell.font = body_regular

    # SECOND ROW OF KPIs
    # blank row
    kpi_sheet["A7"] = ""
    kpi_sheet.row_dimensions[7].height = 30

    # headers
    kpi_sheet.row_dimensions[8].height = 20
    kpi_sheet["A8"] = "BEST LOCATION"
    kpi_sheet["C8"] = "BEST PRODUCT "
    kpi_sheet["E8"] = "PEAK PERIOD"
    for cell in kpi_sheet[8]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[9].height = 20
    kpi_sheet["A9"] = store_sales.index[0][1]  # best location
    kpi_sheet["C9"] = product_sales.index[0]  # best product
    kpi_sheet["E9"] = f"{peak_hour}:00 - {peak_hour + 1}:00"  # peak period

    for cell in kpi_sheet[9]:
        cell.alignment = center_align
        cell.font = body_regular

    # ######## SECOND SHEET - SALES ANALYSIS #######
    sales_analysis = wb.create_sheet(title="Sales Analysis")

    # set columns width
    sales_analysis.column_dimensions['A'].width = 18
    sales_analysis.column_dimensions['B'].width = 18
    sales_analysis.column_dimensions['C'].width = 18
    sales_analysis.column_dimensions['D'].width = 18
    sales_analysis.column_dimensions['E'].width = 18
    sales_analysis.column_dimensions['F'].width = 18
    sales_analysis.column_dimensions['G'].width = 18
    sales_analysis.column_dimensions['H'].width = 18
    sales_analysis.column_dimensions['I'].width = 18

    # set header
    sales_analysis['A1'] = "Sales Analysis"
    sales_analysis['A1'].font = header_18
    sales_analysis.row_dimensions[1].height = 60
    sales_analysis.merge_cells('A1:I1')
    sales_analysis['A1'].alignment = center_align

    # set subheader
    sales_analysis['A2'] = "Revenue performance across products and categories"
    sales_analysis['A2'].font = header_14
    sales_analysis.row_dimensions[2].height = 36
    sales_analysis.merge_cells('A2:I2')
    sales_analysis['A2'].alignment = center_align

    # blank row
    sales_analysis["A3"] = ""
    sales_analysis.row_dimensions[3].height = 30


    #############################################

    # CHART DATA SHEET
    chart_data_ws = wb.create_sheet("Chart Data")

   
    # REVENUE BY CATEGORY
    category_data = category_sales.reset_index()

    category_data.columns = ["Product Category", "Revenue"]

    chart_data_ws["A1"] = "Product Category"
    chart_data_ws["B1"] = "Revenue"

    for row_num, row in enumerate(
        category_data.itertuples(index=False),
        start=2
        ):
        chart_data_ws.cell(row=row_num, column=1, value=row[0])
        chart_data_ws.cell(row=row_num, column=2, value=row[1])

    sales_analysis['A4'] = "Revenue by Product Category"
    sales_analysis['A4'].font = header_14
    sales_analysis.row_dimensions[4].height = 36
    
    category_chart = BarChart()

    category_chart.type = "bar"
    category_chart.style = 13

    category_chart.legend = None

    category_chart.x_axis.majorGridlines = None
    category_chart.y_axis.majorGridlines = None

    category_chart.x_axis.delete = False
    category_chart.y_axis.delete = False

    category_chart.x_axis.majorTickMark = "out"
    category_chart.y_axis.majorTickMark = "out"

    category_chart.x_axis.tickLblPos = "nextTo"
    category_chart.y_axis.tickLblPos = "nextTo"

    category_chart.y_axis.majorUnit = 25000


    category_chart.x_axis.tickLblPos = "low"
    category_chart.x_axis.numFmt = '$#,##0'

    category_data_ref = Reference(
        chart_data_ws,
        min_col=2,
        min_row=1,
        max_row=len(category_data) + 1
    )

    category_labels = Reference(
        chart_data_ws,
        min_col=1,
        min_row=2,
        max_row=len(category_data) + 1
    )

    category_chart.add_data(
        category_data_ref,
        titles_from_data=True
    )

    category_chart.set_categories(category_labels)

    # category_chart.series[0].graphicalProperties.solidFill = "8FD7D7"
    # category_chart.series[0].graphicalProperties.line.solidFill = "8FD7D7"
    

    category_chart.height = 10
    category_chart.width = 15

    sales_analysis.row_dimensions[5].height = category_chart.height * 28.35

    sales_analysis.add_chart(
        category_chart,
        "A5"
    )

    # REVENUE BY MONTH
    monthly_sales = results["sales_by_day"].copy()

    monthly_sales.index = pd.to_datetime(monthly_sales.index)

    monthly_sales = (
        monthly_sales
        .resample("ME")
        .sum()
    )

    monthly_data = monthly_sales.reset_index()

    monthly_data.columns = ["Month", "Revenue"]

    chart_data_ws["D1"] = "Month"
    chart_data_ws["E1"] = "Revenue"

    for row_num, row in enumerate(
        monthly_data.itertuples(index=False),
        start=2
    ):
        chart_data_ws.cell(
            row=row_num,
            column=4,
            value=row[0].strftime("%B")
        )
        chart_data_ws.cell(
            row=row_num,
            column=5,
            value=row[1]
        )

    sales_analysis['F4'] = "Revenue by Month"
    sales_analysis['F4'].font = header_14
    sales_analysis.row_dimensions[4].height = 36

    monthly_chart = LineChart()
    monthly_chart.style = 13
    monthly_chart.legend = None

    monthly_data_ref = Reference(
        chart_data_ws,
        min_col=5,
        min_row=1,
        max_row=len(monthly_data) + 1
    )

    monthly_labels = Reference(
        chart_data_ws,
        min_col=4,
        min_row=2,
        max_row=len(monthly_data) + 1
    )

    monthly_chart.add_data(
        monthly_data_ref,
        titles_from_data=True
    )

    monthly_chart.set_categories(
    monthly_labels
    )

    monthly_chart.x_axis.majorTickMark = "out"
    monthly_chart.y_axis.majorTickMark = "out"

    monthly_chart.x_axis.majorGridlines = None
    monthly_chart.y_axis.majorGridlines = None

    monthly_chart.x_axis.delete = False
    monthly_chart.y_axis.delete = False

    monthly_chart.x_axis.tickLblPos = "low"
    monthly_chart.y_axis.tickLblPos = "nextTo"

    monthly_chart.y_axis.majorUnit = 25000
    monthly_chart.y_axis.numFmt = '$#,##0'

    monthly_chart.series[0].graphicalProperties.solidFill = "EDDCA5"
    monthly_chart.series[0].graphicalProperties.line.solidFill = "EDDCA5"

    monthly_chart.height = 10
    monthly_chart.width = 15

    sales_analysis.row_dimensions[5].height = monthly_chart.height * 28.35

    sales_analysis.add_chart(
        monthly_chart,
        "F5"
    )
    # blank row
    sales_analysis["A3"] = ""
    sales_analysis.row_dimensions[3].height = 30

    # REVENUE BY TIME OF DAY
    # 00B0BE MED TEAL AND C99B38 FOR MED BROWN
    sales_analysis["A7"] = "Revenue by Time of Day"
    sales_analysis["A7"].font = header_14
    sales_analysis.row_dimensions[7].height = 36

    time_sales = results["sales_by_time_of_day"]

    # chart data
    chart_data_ws["G1"] = "Time Period"
    chart_data_ws["H1"] = "Revenue"

    for row_num, (time_period, revenue) in enumerate(
        time_sales.items(),
        start=2
    ):
        chart_data_ws.cell(
            row=row_num,
            column=7,
            value=time_period
        )

        chart_data_ws.cell(
            row=row_num,
            column=8,
            value=revenue
        )

    # create chart
    time_chart = DoughnutChart()

    time_data_ref = Reference(
        chart_data_ws,
        min_col=8,
        min_row=1,
        max_row=len(time_sales) + 1
    )

    time_labels = Reference(
        chart_data_ws,
        min_col=7,
        min_row=2,
        max_row=len(time_sales) + 1
    )

    time_chart.add_data(
        time_data_ref,
        titles_from_data=True
    )

    time_chart.set_categories(time_labels)

    time_chart.height = 10
    time_chart.width = 15
    time_chart.holeSize = 45
    sales_analysis.row_dimensions[8].height = time_chart.height * 28.35

    time_chart.legend.position = "r"

    # add chart
    sales_analysis.add_chart(
        time_chart,
        "A8"
    )

    # TRANSACTIONS PER DAY VS REVENUE BY LOCATION

    sales_analysis["F7"] = "Transactions per Day vs Revenue"
    sales_analysis["F7"].font = header_14
    sales_analysis.row_dimensions[7].height = 36


    # DAILY LOCATION DATA

    daily_location_sales = (   
        quarter_df
        .groupby([
            "Store Location",
            pd.Grouper(key="Transaction Date", freq="W")
        ])
        .agg(
            revenue=("Revenue", "sum"),
            transactions=("Transaction ID", "nunique")
        )
        .reset_index()
    )


    # WRITE CHART DATA

    locations = [
    "Astoria",
    "Hell's Kitchen",
    "Lower Manhattan"
]

    chart_data_ws["M1"] = "Astoria Transactions"
    chart_data_ws["N1"] = "Astoria Revenue"

    chart_data_ws["P1"] = "Hell's Kitchen Transactions"
    chart_data_ws["Q1"] = "Hell's Kitchen Revenue"

    chart_data_ws["S1"] = "Lower Manhattan Transactions"
    chart_data_ws["T1"] = "Lower Manhattan Revenue"


    for col_start, location in zip(
        [13, 16, 19],
        locations
    ):

        location_data = (
            daily_location_sales[
                daily_location_sales["Store Location"] == location
            ]
            .sort_values("Transaction Date")
        )

        for row_num, row in enumerate(
            location_data.itertuples(index=False),
            start=2
        ):

            chart_data_ws.cell(
                row=row_num,
                column=col_start,
                value=row.transactions
            )

            chart_data_ws.cell(
                row=row_num,
                column=col_start + 1,
                value=row.revenue
            )


    # CREATE SCATTER CHART

    scatter_chart = ScatterChart()

    scatter_chart.style = 13

    scatter_chart.x_axis.title = "Transactions per Day"
    scatter_chart.y_axis.title = "Revenue"

    scatter_chart.x_axis.delete = False
    scatter_chart.y_axis.delete = False

    min_transactions = daily_location_sales["transactions"].min()
    max_transactions = daily_location_sales["transactions"].max()

    scatter_chart.x_axis.scaling.min = min_transactions - 10
    scatter_chart.x_axis.scaling.max = max_transactions + 10

    scatter_chart.x_axis.majorTickMark = "out"
    scatter_chart.y_axis.majorTickMark = "out"

    scatter_chart.x_axis.tickLblPos = "low"
    scatter_chart.y_axis.tickLblPos = "nextTo"

    scatter_chart.x_axis.majorGridlines = None
    scatter_chart.y_axis.majorGridlines = None

    scatter_chart.y_axis.numFmt = '$#,##0'

    scatter_chart.height = 10
    scatter_chart.width = 15


    # ADD EACH LOCATION AS A SEPARATE SERIES

    location_columns = [
        (13, 14, "Astoria"),
        (16, 17, "Hell's Kitchen"),
        (19, 20, "Lower Manhattan")
    ]


    for x_col, y_col, location in location_columns:

        location_data = (
            daily_location_sales[
                daily_location_sales["Store Location"] == location
            ]
        )

        x_values = Reference(
            chart_data_ws,
            min_col=x_col,
            min_row=2,
            max_row=len(location_data) + 1
        )

        y_values = Reference(
            chart_data_ws,
            min_col=y_col,
            min_row=2,
            max_row=len(location_data) + 1
        )

        series = Series(
            y_values,
            x_values,
            title=location
        )

        scatter_chart.series.append(series)


    # LEGEND

    scatter_chart.legend.position = "r"


    # ADD CHART

    sales_analysis.add_chart(
        scatter_chart,
        "F8"
    )


    # HEATMAP DATA
    heatmap_start_row = 35

    sales_analysis[f"A{heatmap_start_row}"] = "Revenue by Location, Day and Hour"
    sales_analysis[f"A{heatmap_start_row}"].font = header_14

    sales_analysis.merge_cells(
        start_row=heatmap_start_row,
        start_column=1,
        end_row=heatmap_start_row,
        end_column=9
    )

    sales_analysis[f"A{heatmap_start_row}"].alignment = center_align


    heatmap_row = heatmap_start_row + 2

    for location, heatmap_data in results["revenue_by_day_hour"].items():

        sales_analysis.cell(
            row=heatmap_row,
            column=1,
            value=location
        )

        sales_analysis.cell(
            row=heatmap_row,
            column=1
        ).font = body_bold

        heatmap_row += 1

        # Header row
        sales_analysis.cell(
            row=heatmap_row,
            column=1,
            value="Day"
        )

        for col_num, hour in enumerate(
            heatmap_data.columns,
            start=2
        ):
            sales_analysis.cell(
                row=heatmap_row,
                column=col_num,
                value=f"{hour:02d}:00"
            )

        heatmap_row += 1

        # Data
        for day in heatmap_data.index:

            sales_analysis.cell(
                row=heatmap_row,
                column=1,
                value=day
            )

            for col_num, hour in enumerate(
                heatmap_data.columns,
                start=2
            ):

                value = heatmap_data.loc[day, hour]

                sales_analysis.cell(
                    row=heatmap_row,
                    column=col_num,
                    value=float(value)
                )

            heatmap_row += 1

        heatmap_row += 2


    ######## THIRD SHEET - OBSERVATIONS #######
    observations = wb.create_sheet(title="observations")

    # set columns width
    observations.column_dimensions['A'].width = 20
    observations.column_dimensions['B'].width = 20
    observations.column_dimensions['C'].width = 20
    observations.column_dimensions['D'].width = 20
    observations.column_dimensions['E'].width = 20

    # set header
    observations['A1'] = "Key Observations"
    observations['A1'].font = header_18
    observations.row_dimensions[1].height = 60
    observations.merge_cells('A1:E1')
    observations['A1'].alignment = center_align

    # set subheader
    observations['A3'] = "Automated findings highlighting significant trends and patterns with identified areas for action."
    observations['A3'].font = header_14
    observations.row_dimensions[3].height = 36
    observations.merge_cells('A3:E3')
    observations['A3'].alignment = center_align

    # blank row
    observations["A4"] = ""
    observations.row_dimensions[4].height = 30

    #-------------------------------------------
    observation_start_row = heatmap_row

    sales_analysis.cell(
        row=observation_start_row,
        column=1,
        value="Key Observations"
    )

    sales_analysis.cell(
        row=observation_start_row,
        column=1
    ).font = header_14

    sales_analysis.merge_cells(
        start_row=observation_start_row,
        start_column=1,
        end_row=observation_start_row,
        end_column=9
    )

    sales_analysis.cell(
        row=observation_start_row,
        column=1
    ).alignment = center_align



    ############################################

    # Rank	Product	Revenue
    # 1	    ...	    $...
    # 2	    ...	    $...
    # 3	    ...	    $...
    
    # TOP LOCATION
    # Lower Manhattan

    # REVENUE
    # $XXX,XXX

    # SHARE OF TOTAL
    # XX.X%
    # '''


    # hide helper sheets
    chart_data_ws.sheet_state = "hidden"

    # back to first sheet
    # wb.active = kpi_sheet
    wb.active = sales_analysis

    # Save the file
    report_name = (
    f"Performance Report - "
    f"{quarter} - {start_month} – {end_month} {year}.xlsx"
    )
    #------------------------------------
    print(f"Saving report: {report_name}")
    wb.save(report_name)

Saving report: Performance Report - 2023Q1 - January – March 2023.xlsx
Saving report: Performance Report - 2023Q2 - April – June 2023.xlsx
